<a href="https://colab.research.google.com/github/lipchenko/machine_learning_lipchenko/blob/main/my_w2v_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [2]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 51.4 MB/s eta 0:00:00


## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [3]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

In [4]:
import gensim.downloader as api

# Получить информацию обо всех доступных ресурсах
info = api.info()

# Вывести названия всех предобученных моделей
print("Доступные модели:")
for model_name in info['models'].keys():
    print(f" - {model_name}")

Доступные модели:
 - fasttext-wiki-news-subwords-300
 - conceptnet-numberbatch-17-06-300
 - word2vec-ruscorpora-300
 - word2vec-google-news-300
 - glove-wiki-gigaword-50
 - glove-wiki-gigaword-100
 - glove-wiki-gigaword-200
 - glove-wiki-gigaword-300
 - glove-twitter-25
 - glove-twitter-50
 - glove-twitter-100
 - glove-twitter-200
 - __testing_word2vec-matrix-synopsis


1) fasttext-wiki-news-subwords-300

Источник данных - википедия, новости
объем векторов - 300, содержит около 1 миллиона слов
датасет может подойти для обработки редких слов или слов с ошибками

2) conceptnet-numberbatch-17-06-300

Источник данных - текстовые корпуса, корпуса субтитров для разных языков
объем векторов - 300, для английского языка содержит 516 тысяч слов. всего содержит 9 языков.
датасет может подойти для сравнения слов в разных языках и статистических исследований и закономерностей в разных языках.

3) glove-wiki-gigaword-200

Источник данных - википедия и гигаворд
объем векторов - 200, для английского языка содержит 516 тысяч слов. всего содержит 9 языков.
датасет может подойти для поиска семантически близких слов, кластеризации и классификации текстов.


**Базовые операции с векторами**

In [5]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [6]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [ ]:
w2v_model = api.load('glove-twitter-200')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 758.5/758.5MB downloaded
Размер словаря: 1193514
Размерность векторов: 200


In [17]:
# Проверим, загружена ли модель и есть ли слова
print("Переменная w2v_model определена?", 'w2v_model' in globals())
if 'w2v_model' in globals():
    print("Размер словаря:", len(w2v_model.key_to_index))
    print("Пример слова 'dog':", 'dog' in w2v_model.key_to_index)

Переменная w2v_model определена? True
Размер словаря: 400000
Пример слова 'dog': True


2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [19]:
def get_similar_words(word, topn=10):
    word = word.lower().strip()
    if word in w2v_model.key_to_index:
        similar_words = w2v_model.most_similar(word, topn=topn)
        return similar_words
    else:
        return None

if __name__ == "__main__":
    word = input("Введите нужное слово: ").strip().lower()
    similar = get_similar_words(word, topn=10)
    if similar:
        print(f"Слова, похожие на '{word}':")
        for sim_word, score in similar:
            print(f"  {sim_word}: {score:.4f}")
    else:
        print("Слово не найдено в модели.")


Введите нужное слово: dog
Слова, похожие на 'dog':
  cat: 0.8798
  dogs: 0.8344
  pet: 0.7450
  puppy: 0.7236
  horse: 0.7110
  animal: 0.6817
  pig: 0.6554
  boy: 0.6545
  cats: 0.6472
  rabbit: 0.6469


3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [20]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [22]:
skipgram_model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,      # размерность векторов
    window=3,             # размер контекстного окна
    min_count=1,          # минимальная частота слова
    workers=2,            # количество ядер
    sg=1                  # 1 = Skip-Gram
)

print(f"Размер словаря: {len(skipgram_model.wv.key_to_index)}")

Размер словаря: 65


In [23]:
print(f"Слова в словаре: {list(w2v_model.key_to_index.keys())[:10]}...")

Слова в словаре: ['the', ',', '.', 'of', 'to', 'and', 'in', 'a', '"', "'s"]...


4. Проверьте модель

In [24]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = skipgram_model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  вино: 0.2398
  ингредиенты: 0.2172
  хлеб: 0.1938
  брокколи: 0.1846
  кипятить: 0.1711


In [25]:
similar_words = skipgram_model.wv.most_similar('духовка', topn=5)
print("Слова, похожие на 'духовка':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

similar_words = skipgram_model.wv.most_similar('овощи', topn=5)
print("Слова, похожие на 'овощи':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'духовка':
  ингредиенты: 0.3199
  десерт: 0.3064
  холодильник: 0.2705
  питание: 0.2243
  пирог: 0.2142
Слова, похожие на 'овощи':
  мариновать: 0.2716
  хлеб: 0.2691
  гриль: 0.2546
  фольга: 0.2409
  сахар: 0.2108


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [26]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [32]:
def sim_word(word):
    print(f"\nСравнение для слова: '{word}'")
    try:
        similar = ft_model.wv.most_similar(word, topn=5)
        print(f"Слова, похожие на '{word}':")
        for similar_word, score in similar:
            print(f"  {similar_word}: {score:.4f}")
    except KeyError:
        print(f"Слово '{word}' не найдено в словаре")


sim_word('варить')
sim_word('духовка')
sim_word('овощи')


Сравнение для слова: 'варить'
Слова, похожие на 'варить':
  жарить: 0.5353
  парить: 0.4805
  месить: 0.3541
  тушить: 0.3405
  специи: 0.2622

Сравнение для слова: 'духовка'
Слова, похожие на 'духовка':
  взбивать: 0.4565
  лимон: 0.3561
  салат: 0.3050
  курица: 0.3041
  тост: 0.2944

Сравнение для слова: 'овощи'
Слова, похожие на 'овощи':
  жарить: 0.2960
  фольга: 0.2574
  морковь: 0.2297
  соус: 0.2172
  торт: 0.2094


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [30]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = skipgram_model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models('lirning')
compare_models('nural')
compare_models('glopal')


Сравнение для слова: 'lirning'
  Word2Vec: слово не найдено
  FastText: ['паста', 'тушить']

Сравнение для слова: 'nural'
  Word2Vec: слово не найдено
  FastText: ['гриль', 'чашка']

Сравнение для слова: 'glopal'
  Word2Vec: слово не найдено
  FastText: ['месить', 'картофель']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [34]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [35]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [36]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [37]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


*8*. Сравните схожесть doc_2 и doc_4

In [41]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(2, 4)  # machine learning vs deep learning

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [ ]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_1"]
print(f"Вектор документа doc_1: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_1", topn=2)
print("\nДокументы, похожие на doc_1:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_1: [-0.00435809 -0.00022557 -0.01330682 -0.01307992 -0.00394024]...

Документы, похожие на doc_1:
  doc_0: 0.2735
    Текст: machine learning is interesting
  doc_3: 0.2031
    Текст: artificial intelligence is amazing


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [45]:

cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]


tagged_docs = [TaggedDocument(words=sent, tags=[i])
for i, sent in enumerate(cooking_sentences)]

test_words = ['варить', 'духовка', 'овощи']
topn = 5

def show_similar_words(model, model_name):
    print(f"Модель Doc2Vec, размерность {model_name}")
    for word in test_words:
        try:
            similar = model.wv.most_similar(word, topn=topn)
            print(f"Слова, похожие на '{word}':")
            for sim_word, score in similar:
                print(f"  {sim_word}: {score:.4f}")
        except KeyError:
            print(f"Слово '{word}' не найдено в словаре модели {model_name}")

vector_sizes = [10, 50, 100]

for size in vector_sizes:
    model = Doc2Vec(
        documents=tagged_docs,
        vector_size=size,
        window=3,
        min_count=1,
        workers=2,
        epochs=20,
        seed=42
    )
    print(f"Обучена модель с vector_size={size}")
    print(f"Количество документов: {len(model.dv.key_to_index)}")
    print(f"Размер словаря слов: {len(model.wv.key_to_index)}")


    show_similar_words(model, size)

Обучена модель с vector_size=10
Количество документов: 0
Размер словаря слов: 65
Модель Doc2Vec, размерность 10
Слова, похожие на 'варить':
  хлеб: 0.8298
  дрожжи: 0.6745
  духовка: 0.6086
  сахар: 0.5632
  сливки: 0.5571
Слова, похожие на 'духовка':
  дрожжи: 0.6450
  салат: 0.6349
  хлеб: 0.6213
  варить: 0.6086
  резать: 0.5687
Слова, похожие на 'овощи':
  бекон: 0.7320
  парить: 0.7173
  горшок: 0.4796
  огурцы: 0.4434
  гриль: 0.4307
Обучена модель с vector_size=50
Количество документов: 0
Размер словаря слов: 65
Модель Doc2Vec, размерность 50
Слова, похожие на 'варить':
  горшок: 0.2117
  резать: 0.2099
  салат: 0.2002
  соус: 0.1900
  картофель: 0.1869
Слова, похожие на 'духовка':
  барбекю: 0.2805
  мариновать: 0.2652
  молоко: 0.2301
  специи: 0.2259
  жарить: 0.1878
Слова, похожие на 'овощи':
  вино: 0.3321
  сковорода: 0.2638
  взбивать: 0.2416
  ингредиенты: 0.2326
  рыба: 0.2129
Обучена модель с vector_size=100
Количество документов: 0
Размер словаря слов: 65
Модель Doc2V